In [0]:
from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data
from fraud_mlops.azureml_retraining import submit_model_pipeline

TRAINING_DATA_PATH = dbutils.jobs.taskValues.get(
    taskKey='export_retraining_snapshot',
    key='training_data_path'
)

SNAPSHOT_ID = dbutils.jobs.taskValues.get(
    taskKey='build_retraining_dataset',
    key='snapshot_id'
)

print(f"Training data path: {TRAINING_DATA_PATH}")
print(f"Snapshot ID: {SNAPSHOT_ID}")

VOLUME_PREFIX = '/Volumes/fraud_mlops/gold/training_snapshots/'
ADLS_PREFIX = 'abfss://fraudmlops@fraudmlopsstorage.dfs.core.windows.net/training_snapshots/'

if TRAINING_DATA_PATH.startswith(VOLUME_PREFIX):
    relative_path = TRAINING_DATA_PATH[len(VOLUME_PREFIX):]

    AZURE_DATA_PATH = ADLS_PREFIX + relative_path

elif TRAINING_DATA_PATH.startswith('abfss://'):
    AZURE_DATA_PATH = TRAINING_DATA_PATH

else:
    raise ValueError(f'Unsupported training data path:{TRAINING_DATA_PATH}')

print("Azure ML data path:", AZURE_DATA_PATH)

credential = dbutils.credentials.getServiceCredentialsProvider('fraud_aml_managed_identity')

AZURE_ML_WORKSPACE = 'aml-fraud-mlops'
ml_client = MLClient(
    credential=credential,
    subscription_id='a04797d3-24a7-4263-b445-298c1dc326c5',
    resource_group_name='rg-fraud-mlops',
    workspace_name= AZURE_ML_WORKSPACE  
)

print(
    "Connected to Azure ML workspace:",
    AZURE_ML_WORKSPACE
)

DATA_ASSET_NAME = 'paysim-fraud-training-snapshot'
DATA_ASSET_VERSION = SNAPSHOT_ID.replace('-', '')
print(
    "New Azure ML data version:",
    DATA_ASSET_VERSION
)

training_data_asset = Data(
    path=AZURE_DATA_PATH,
    type=AssetTypes.URI_FOLDER,
    name=DATA_ASSET_NAME,
    description='Automatically generated fraud retraining snapshot from Databricks.',
    version=DATA_ASSET_VERSION,
    tags={
        "retraining_snapshot_id":
            SNAPSHOT_ID,

        "source":
            "databricks_ct_workflow"
    }
)

registered_data=ml_client.data.create_or_update(training_data_asset)


DATA_ASSET_REFERENCE = (
    f"azureml:"
    f"{registered_data.name}:"
    f"{registered_data.version}"
)
print(
    "Registered Azure ML data asset:",
    DATA_ASSET_REFERENCE
)

print(
    "Starting Azure ML retraining..."
)

result = submit_model_pipeline(
    ml_client=ml_client,
    data_path=DATA_ASSET_REFERENCE,
    data_version_tag=(
        f"{DATA_ASSET_NAME}:"
        f"{DATA_ASSET_VERSION}"
    )
)

# ---------------------------------------------------------
# GET RESULT
# ---------------------------------------------------------

pipeline_job_id = (
    result["pipeline_job_id"]
)

model_name = (
    result["model_name"]
)

model_version = (
    result["model_version"]
)

print(
    "Azure ML pipeline:",
    pipeline_job_id
)

print(
    "Registered challenger:",
    f"{model_name}:{model_version}"
)

# ---------------------------------------------------------
# PUBLISH OUTPUTS FOR LATER TASKS / CD
# ---------------------------------------------------------

dbutils.jobs.taskValues.set(
    key="azureml_pipeline_job",
    value=str(pipeline_job_id)
)

dbutils.jobs.taskValues.set(
    key="registered_model_version",
    value=str(model_version)
)

dbutils.jobs.taskValues.set(
    key="azureml_data_asset",
    value=DATA_ASSET_REFERENCE
)

# ---------------------------------------------------------
# COMPLETION MESSAGE
# ---------------------------------------------------------

print(
    "\nContinuous Training completed successfully."
)

print(
    "Snapshot ID:",
    SNAPSHOT_ID
)

print(
    "Azure ML data asset:",
    DATA_ASSET_REFERENCE
)

print(
    "Azure ML pipeline:",
    pipeline_job_id
)

print(
    "New challenger:",
    f"{model_name}:{model_version}"
)


